In [6]:
import os
import re
import pandas as pd
import tkinter as tk
from tkinter import filedialog

In [7]:
def consolidar_resultados_publicacion(directorio_raiz):
    """
    Recorre carpetas buscando '_resumen.txt', extrae datos, 
    escala las áreas por 10^11 y los devuelve en inglés.
    """
    datos_recopilados = []
    
    print(f"\n🔍 Scanning directory: {directorio_raiz}...\n")

    for root, dirs, files in os.walk(directorio_raiz):
        for file in files:
            if file.endswith("_resumen.txt"):
                ruta_completa = os.path.join(root, file)
                nombre_planeta = os.path.basename(root).replace("_", " ")
                
                with open(ruta_completa, 'r', encoding='utf-8') as f:
                    contenido = f.read()
                
                try:
                    # Extraer datos de conteo con Regex (Zonas)
                    z_estables = int(re.search(r'Total estables:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                    z_inestables = int(re.search(r'Total inestables:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                    z_col_m1 = int(re.search(r'Total colisi[oó]n m1:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                    z_col_m2 = int(re.search(r'Total colisi[oó]n m2:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                    
                    # Extraer datos de áreas
                    a_est = float(re.search(r'[AÁaá]rea estable:\s*([0-9\.eE\+\-]+)', contenido).group(1))
                    a_inest = float(re.search(r'[AÁaá]rea inestable:\s*([0-9\.eE\+\-]+)', contenido).group(1))
                    a_col_m2 = float(re.search(r'[AÁaá]rea de colisi[oó]n con m2:\s*([0-9\.eE\+\-]+)', contenido).group(1))
                    
                    # ESCALAR POR 10^11
                    FACTOR_ESCALA = 1e11
                    
                    # Agregar diccionario a la lista (En inglés)
                    datos_recopilados.append({
                        "Planet": nombre_planeta,
                        "Stable Zones": z_estables,
                        "Unstable Zones": z_inestables,
                        "Col. m1 Zones": z_col_m1,
                        "Col. m2 Zones": z_col_m2,
                        "Stable Area": a_est / FACTOR_ESCALA,
                        "Unstable Area": a_inest / FACTOR_ESCALA,
                        "Col. m2 Area": a_col_m2 / FACTOR_ESCALA
                    })
                    print(f"✅ Data extracted from: {nombre_planeta}")
                except Exception as e:
                    print(f"❌ Error processing {file}: {e}")
                    
    # Retornar el DataFrame creado (Línea corregida)
    return pd.DataFrame(datos_recopilados)

In [8]:
def generar_archivos_publicacion(df, nombre_salida="Global_Results_Table"):
    """
    Exporta la tabla a formatos útiles (CSV, TXT, LaTeX).
    """
    if df.empty:
        print("The table is empty. Nothing was exported.")
        return

    # 1. Exportar a CSV (Mantiene columnas sencillas para datos puros)
    df.to_csv(f"{nombre_salida}.csv", index=False)
    
    # 2. Exportar a TXT (Formateado a 2 decimales)
    with open(f"{nombre_salida}.txt", "w", encoding='utf-8') as f:
        f.write(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
        
    # 3. Exportar a LaTeX 
    df_latex = df.copy()
    
    # Renombrar columnas para agregar la notación matemática en la cabecera
    df_latex.rename(columns={
        "Stable Area": "Stable Area ($10^{11}$ km$^2$)",
        "Unstable Area": "Unstable Area ($10^{11}$ km$^2$)",
        "Col. m2 Area": "Col. m2 Area ($10^{11}$ km$^2$)"
    }, inplace=True)
    
    try:
        # Método moderno de Pandas (>= 1.3)
        latex_code = df_latex.style.format(precision=2).hide(axis="index").to_latex(
            hrules=True, 
            caption="Orbital stability analysis and corresponding areas.", 
            label="tab:stability_areas",
            column_format="lrrrrrrr"
        )
    except AttributeError:
        # Método antiguo de Pandas (< 1.3)
        latex_code = df_latex.to_latex(
            index=False,
            float_format="%.2f",
            column_format="lrrrrrrr",
            caption="Orbital stability analysis and corresponding areas.",
            label="tab:stability_areas",
            escape=False # Permite que se imprima el $10^{11}$ sin desconfigurarse
        )
        
    with open(f"{nombre_salida}.tex", "w", encoding='utf-8') as f:
        f.write(latex_code)
        
    print(f"\n🎉 Table exported successfully! Created: .csv, .txt and .tex")

In [9]:
if __name__ == "__main__":
    # Ocultar la ventana principal gris de Tkinter para que solo salga el cuadro de diálogo
    root = tk.Tk()
    root.withdraw()
    
    # Forzar que la ventana aparezca por encima de otras aplicaciones
    root.attributes('-topmost', True)

    print("Esperando a que selecciones una carpeta en la ventana emergente...")
    
    # Abrir la ventana para seleccionar directorio
    # Abrir la ventana para seleccionar directorio en la ruta específica
    DIRECTORIO_RAIZ = filedialog.askdirectory(initialdir=r'C:\Users\darkm\Desktop\archivos_Investigación\Investigacion\Troyanos\Resultados_Estrellas_K', title="Selecciona la carpeta raíz")
    # Verificar si el usuario realmente seleccionó una carpeta o si le dio a "Cancelar"
    if DIRECTORIO_RAIZ:
        print(f"Carpeta seleccionada con éxito: {DIRECTORIO_RAIZ}")
        
        tabla_final = consolidar_resultados_publicacion(DIRECTORIO_RAIZ)
        
        if not tabla_final.empty:
            print("\n=== VISTA PREVIA DE LA TABLA ===")
            print(tabla_final.to_string(index=False, float_format=lambda x: f"{x:.2e}"))
            
            # Generar los archivos en el mismo lugar desde donde corres el código
            generar_archivos_publicacion(tabla_final)
        else:
            print("\n No se encontraron archivos '_resumen.txt' válidos en esa carpeta.")
    else:
        print("\n Operación cancelada. No seleccionaste ninguna carpeta.")

Esperando a que selecciones una carpeta en la ventana emergente...
Carpeta seleccionada con éxito: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_F

🔍 Scanning directory: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_F...

✅ Data extracted from: DMPP-1 b
✅ Data extracted from: DMPP-1 c
✅ Data extracted from: DMPP-1 d
✅ Data extracted from: DMPP-1 e
✅ Data extracted from: DMPP-2 b
✅ Data extracted from: DMPP-2 c
✅ Data extracted from: DMPP-2 d
✅ Data extracted from: HD 142 A d
✅ Data extracted from: HD 142 b
✅ Data extracted from: HD 22946 d
✅ Data extracted from: TOI-411 b
✅ Data extracted from: TOI-411 c
✅ Data extracted from: HD 28109 b
✅ Data extracted from: HD 28109 c
✅ Data extracted from: HD 28109 d
✅ Data extracted from: HD 50554 c
✅ Data extracted from: HD 50554 d
✅ Data extracted from: HD 73344 b
✅ Data extracted from: HD 73344 c
✅ Data extracted from: HIP 41378 b
✅ Data extracted from: HIP 